In [1]:
import numpy as np
import trimesh

# --- CONFIG ---
gro_path = "./1mol.gro"                   
sphere_radius_scale = 1.5                 # balls (atoms): 2.0 × van der Waals radius   # idk why 1.5 :(
bond_radius = 0.1                        # sticks (bonds): cylinder radius in Å
sphere_subdiv = 2                         # atom sphere detail (2 is moderate)

### Import data and Setup

In [ ]:
def parse_gro(path, atoms_per_mol=84):
    """
    Parse a .gro file with multiple molecules of equal atom count.

    Parameters
    ----------
    path : str
        Path to .gro file
    atoms_per_mol : int
        Number of atoms per molecule (default 84)

    Returns
    -------
    molecules : dict[int, list[tuple]]
        Dictionary of molecules:
          molecules[i] = [(atomname, coord), ...] for atom coords in Å
        where i = 1..N_molecules
    """
    with open(path, "r") as f:
        _title = f.readline()
        n = int(f.readline().strip())  # total number of atoms
        lines = [f.readline() for _ in range(n)]
        box_line = f.readline().strip()  # box line

    # --- parse atom coordinates ---
    n_mol = n // atoms_per_mol  # determine number of molecules using integer division
    if n % atoms_per_mol != 0:
        raise ValueError(f"Total atoms {n} not divisible by atoms_per_mol={atoms_per_mol}")

    molecules = {}
    for m in range(n_mol):
        start = m * atoms_per_mol
        end = start + atoms_per_mol
        mol_atoms = []
        for line in lines[start:end]:
            # parse atom name and coords
            atomname = line[10:15].strip()
            x = float(line[20:28]) * 10.0  # nm → Å
            y = float(line[28:36]) * 10.0
            z = float(line[36:44]) * 10.0
            mol_atoms.append((atomname, np.array([x, y, z], dtype=float)))
        molecules[m+1] = mol_atoms

    # --- parse box dimensions ---
    box_vals = [float(x) for x in box_line.split()]
    if len(box_vals) == 3:
        # orthorhombic box
        box = np.array(box_vals) * 10.0  # Å
    elif len(box_vals) == 9:
        # triclinic box: x, y, z vectors in nm
        box = np.array(box_vals).reshape(3, 3) * 10.0  # Å
    else:
        raise ValueError(f"Unexpected box format with {len(box_vals)} values")

    return molecules, box


In [3]:
molecules, box = parse_gro("../../data/npt-HK4.gro", atoms_per_mol=84)

In [4]:
def infer_element(atomname):
    # common water aliases
    if atomname in ("OW", "HW", "HW1", "HW2"): 
        return "O" if atomname=="OW" else "H"
    
    # simple: first letter, capitalize second if lowercase
    a = ''.join([c for c in atomname if c.isalpha()])   
    
    # join() joins items in an iterable into one string, '' is specified as the separator.
    # isalpha() method returns True if all the characters are alphabet letters (a-z).

    if a == '': 
        return "C"

    if len(a) >= 2 and a[1].islower(): 
        return (a[0]+a[1]).capitalize()
    
    return a[0].upper()

In [5]:
# --- Radii (Å) ---
vdw = {"H":1.20,"C":1.70,"N":1.55,"O":1.52,"F":1.47,"P":1.80,"S":1.80,"Cl":1.75,"Na":2.27,"K":2.75,"Ca":2.31}
cov = {"H":0.31,"C":0.76,"N":0.71,"O":0.66,"F":0.57,"P":1.07,"S":1.05,"Cl":1.02,"Na":1.66,"K":2.03,"Ca":1.74}

### Build molecules model

In [6]:
# --- Minimum Image Convention (vector) ---
def mic_vector(dx, box_length):
    """Return minimum-image displacement for vector dx under PBC."""
    return dx - np.rint(dx / box_length) * box_length

def mic_distance(a, b, box_length):
    """Return MIC distance between two 3D points a,b."""
    return np.linalg.norm(mic_vector(b - a, box_length))

In [ ]:
def build_molecule_ballstick(coords, elements,
                             vdw, cov,
                             box_length,
                             sphere_radius_scale=0.3,
                             sphere_subdiv=2,
                             bond_radius=0.1):
    """
    Build trimesh ball-and-stick model under periodic boundary conditions.
    """
    # radii arrays
    vdw_r = np.array([vdw.get(e, 1.70) for e in elements])
    cov_r = np.array([cov.get(e, 0.77) for e in elements])

    meshes = []

    # --- Atoms as spheres (wrapped into primary box [0,L)) ---
    coords_wrapped = np.mod(coords, box_length)
    for pos, r in zip(coords_wrapped, vdw_r * sphere_radius_scale):
        sph = trimesh.creation.icosphere(subdivisions=sphere_subdiv, radius=float(r))
        sph.apply_translation(pos)
        meshes.append(sph)

    # --- Bonds (MIC criterion with covalent radii) ---
    n = len(coords)
    for i in range(n):
        for j in range(i+1, n):
            d = mic_distance(coords[i], coords[j], box_length)
            thr = 1.2 * (cov_r[i] + cov_r[j])
            if d < thr:
                # Unwrap j relative to i
                disp = mic_vector(coords[j] - coords[i], box_length)
                pos_i = np.mod(coords[i], box_length)
                pos_j = pos_i + disp  # may fall outside box but correct bond vector
                seg = np.vstack((pos_i, pos_j))
                cyl = trimesh.creation.cylinder(radius=bond_radius,
                                                segment=seg, sections=24)
                meshes.append(cyl)

    # --- Merge all into one mesh ---
    molecule = trimesh.util.concatenate(meshes)
    return molecule


In [8]:
def molecules_to_meshes(molecules, box,
                        vdw, cov,
                        sphere_radius_scale=0.3,
                        sphere_subdiv=2,
                        bond_radius=0.1):
    """
    Convert parsed molecules into trimesh meshes.

    Parameters
    ----------
    molecules : dict[int, list[tuple]]
        From parse_gro(): molecules[i] = [(atomname, coords), ...]
        coords must be in Å
    box : np.ndarray
        Simulation box (Å), shape (3,) for orthorhombic or (3,3) for triclinic
    vdw, cov : dict
        Van der Waals and covalent radii
    sphere_radius_scale : float
        Scaling factor for atom radii
    sphere_subdiv : int
        Subdivisions for icosphere (mesh resolution)
    bond_radius : float
        Cylinder radius for bonds

    Returns
    -------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary of molecule meshes keyed by mol_id
    """
    mol_meshes = {}
    # assume orthorhombic box for now
    if box.shape == (3,):
        box_length = box
    else:
        raise NotImplementedError("Triclinic box handling not yet implemented")

    for mol_id, atoms in molecules.items():
        elements = [infer_element(name) for name, _ in atoms]
        coords = np.vstack([pos for _, pos in atoms])  # (n_atoms, 3)

        mesh = build_molecule_ballstick(
            coords, elements, vdw, cov, box_length,
            sphere_radius_scale=sphere_radius_scale,
            sphere_subdiv=sphere_subdiv,
            bond_radius=bond_radius
        )
        mol_meshes[mol_id] = mesh

    return mol_meshes


In [9]:
# Build meshes for all molecules
mol_meshes = molecules_to_meshes(molecules, box, vdw, cov,
                                 sphere_radius_scale=sphere_radius_scale,
                                 sphere_subdiv=sphere_subdiv,
                                 bond_radius=bond_radius
                                 )

### Nearest neighbors

In [ ]:
def mic_displacement(vec, box):
    box = np.array(box, dtype=float)
    return vec - box * np.round(vec / box)

def compute_centroids_and_radii_pbc(mol_meshes, box):
    """
    Compute PBC-aware centroids and radii for molecules.
    
    Steps:
    1. Choose a reference vertex per molecule.
    2. MIC-displace all other vertices relative to it (unwrap locally).
    3. Compute centroid of unwrapped coordinates.
    4. MIC distances from centroid -> vertices = radius.
    """
    centroids, radii = {}, {}
    box = np.array(box, dtype=float)

    for mol_id, mesh in mol_meshes.items():
        verts = mesh.vertices
        ref = verts[0]  # reference atom
        disp = mic_displacement(verts - ref, box)
        unwrapped = ref + disp

        # centroid in unwrapped space
        center = unwrapped.mean(axis=0)

        # MIC distances from centroid to each vertex
        disp_center = mic_displacement(verts - center, box)
        radius = np.linalg.norm(disp_center, axis=1).max()

        centroids[mol_id] = center
        radii[mol_id] = radius

    return centroids, radii


In [14]:
centroids, radii = compute_centroids_and_radii_pbc(mol_meshes, box)

In [ ]:
def blocked_by_any(i, j, centroids, radii, mol_meshes, kd, ids, box, cutoff_margin=2.0):
    ci, cj = centroids[i], centroids[j]
    seg_vec = cj - ci
    seg_len = np.linalg.norm(seg_vec)
    if seg_len < 1e-6:
        return False
    direction = seg_vec / seg_len
    midpt = (ci + cj) / 2.0

    # Query potential blockers (MODIFY THIS PART)
    search_radius = seg_len / 2 + max(radii.values()) + cutoff_margin
    cand_idx = kd.query_ball_point(midpt, r=search_radius)

    for kdx in cand_idx:
        mol_k = ids[kdx]   # map index back to mol_id
        if mol_k in (i, j):
            continue

        # Quick sphere reject
        ck = centroids[mol_k]
        v = cj - ci
        w = ck - ci
        proj = np.dot(w, v) / np.dot(v, v)
        proj = np.clip(proj, 0.0, 1.0)
        closest = ci + proj * v
        if np.linalg.norm(ck - closest) > radii[mol_k]:
            continue

        # Expensive ray test
        if mol_meshes[mol_k].ray.intersects_any(
            ray_origins=ci.reshape(1, 3),
            ray_directions=direction.reshape(1, 3)
        ):
            return True
    return False

In [16]:
from scipy.spatial import cKDTree
from tqdm import tqdm

# --- utility: wrap points into [0, L)
def wrap_points(points, box):
    box = np.array(box, dtype=float)
    return points % box

# # --- main neighbor finder ---
# def find_neighbors(mol_meshes, box, pair_cutoff=10.0, cutoff_margin=2.0, show_progress=True):
#     centroids, radii = compute_centroids_and_radii_pbc(mol_meshes, box)

#     ids = list(centroids.keys())
#     coords = np.vstack([centroids[i] for i in ids])

#     # wrap into box before KD-tree
#     coords_wrapped = wrap_points(coords, box)

#     kd = cKDTree(coords_wrapped, boxsize=box)

#     neighbors = []
#     iterator = enumerate(ids)
#     if show_progress:
#         iterator = tqdm(iterator, total=len(ids), desc="Neighbor search")

#     for idx, i in iterator:
#         ci = coords_wrapped[idx]  # use wrapped centroid for query
#         cand_idx = kd.query_ball_point(ci, r=pair_cutoff)

#         for jdx in cand_idx:
#             j = ids[jdx]
#             if j <= i:
#                 continue
#             # check blockers with unwrapped centroids + MIC
#             if not blocked_by_any(i, j, centroids, radii, mol_meshes, kd, ids, box, cutoff_margin):
#                 neighbors.append((i, j))

#     return neighbors


In [ ]:
# neighbors = find_neighbors(mol_meshes, box, pair_cutoff=10.0, cutoff_margin=2.0, show_progress=True)

Neighbor search: 100%|██████████| 1501/1501 [04:09<00:00,  6.02it/s] 


In [ ]:
# len(neighbors)

569

In [18]:
ids = list(centroids.keys())
coords = np.vstack([centroids[i] for i in ids])

# wrap into box before KD-tree
coords_wrapped = wrap_points(coords, box)

kd = cKDTree(coords_wrapped, boxsize=box)

In [130]:
help(cKDTree)

Help on class cKDTree in module scipy.spatial._ckdtree:

class cKDTree(builtins.object)
 |  cKDTree(data, leafsize=16, compact_nodes=True, copy_data=False,
 |          balanced_tree=True, boxsize=None)
 |
 |  kd-tree for quick nearest-neighbor lookup
 |
 |  This class provides an index into a set of k-dimensional points
 |  which can be used to rapidly look up the nearest neighbors of any
 |  point.
 |
 |  .. note::
 |     `cKDTree` is functionally identical to `KDTree`. Prior to SciPy
 |     v1.6.0, `cKDTree` had better performance and slightly different
 |     functionality but now the two names exist only for
 |     backward-compatibility reasons. If compatibility with SciPy < 1.6 is not
 |     a concern, prefer `KDTree`.
 |
 |  Parameters
 |  ----------
 |  data : array_like, shape (n,m)
 |      The n data points of dimension m to be indexed. This array is
 |      not copied unless this is necessary to produce a contiguous
 |      array of doubles, and so modifying this data will r

In [46]:
neighbor_pairs = []

for i in range(1,1501):
    if not blocked_by_any(42, i, centroids, radii, mol_meshes, kd, ids, box=box, cutoff_margin=2.0):
        neighbor_pairs.append((42,i))

print(neighbor_pairs)

[(42, 42), (42, 337), (42, 490), (42, 663), (42, 746), (42, 1131), (42, 1141)]


In [23]:
def find_neighbors(centroids, radii, mol_meshes, kd, ids, box):
    neighbor_pairs = []

    for i in range(1, 3):
        for j in range(i+1, 1501):
            if not blocked_by_any(i, j, centroids, radii, mol_meshes, kd, ids, box, cutoff_margin=2.0):
                neighbor_pairs.append((i, j))

    return neighbor_pairs


In [24]:
find_neighbors(centroids, radii, mol_meshes, kd, ids, box)

[(1, 4),
 (1, 471),
 (1, 519),
 (1, 704),
 (1, 782),
 (1, 847),
 (1, 1054),
 (2, 3),
 (2, 18),
 (2, 21),
 (2, 44),
 (2, 69),
 (2, 147),
 (2, 160),
 (2, 231),
 (2, 274),
 (2, 532),
 (2, 612),
 (2, 637),
 (2, 645),
 (2, 670),
 (2, 770),
 (2, 788),
 (2, 877),
 (2, 878),
 (2, 882),
 (2, 984),
 (2, 1084),
 (2, 1366),
 (2, 1441)]

In [ ]:
def nearest_neighbors(mol_meshes, box, centroids, k=10, return_meshes=False):
    """
    Find the k nearest neighbors of each molecule, based on centroid distance.
    
    Parameters
    ----------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary of molecule meshes.
    box : float or array-like
        Simulation box length (for PBC). Pass a scalar if cubic.
    k : int
        Number of nearest neighbors to return per molecule.

    Returns
    -------
    dict[int, list[tuple[int,float]]]
        Mapping mol_id -> list of (neighbor_id, distance).
    """

    ids = list(centroids.keys())
    coords = np.vstack([centroids[i] for i in ids])
    coords_wrapped = wrap_points(coords, box)
    kd = cKDTree(coords_wrapped, boxsize=box)

    neighbors = {}
    for idx, mol_id in enumerate(ids):
        dists, idxs = kd.query(coords[idx], k=k+1)
        dists, idxs = dists[1:], idxs[1:]
        if return_meshes:
            neighbors[mol_id] = [(mol_meshes[ids[j]], float(d)) for j, d in zip(idxs, dists)]
        else:
            neighbors[mol_id] = [(ids[j], float(d)) for j, d in zip(idxs, dists)]
    return neighbors


In [132]:
neighbors = nearest_neighbors(mol_meshes, box, centroids, k=10, return_meshes=False)


In [133]:
neighbor_candidates = [(key, t[0]) for key, value in neighbors.items() for t in value]

In [134]:
neighbor_candidates

[(1, 1308),
 (1, 2),
 (1, 244),
 (1, 847),
 (1, 943),
 (1, 4),
 (1, 1088),
 (1, 1283),
 (1, 18),
 (1, 359),
 (2, 4),
 (2, 18),
 (2, 704),
 (2, 1),
 (2, 780),
 (2, 127),
 (2, 226),
 (2, 1206),
 (2, 3),
 (2, 984),
 (3, 704),
 (3, 1380),
 (3, 258),
 (3, 1461),
 (3, 1014),
 (3, 518),
 (3, 847),
 (3, 147),
 (3, 1474),
 (3, 2),
 (4, 2),
 (4, 984),
 (4, 1330),
 (4, 1),
 (4, 1206),
 (4, 244),
 (4, 359),
 (4, 18),
 (4, 226),
 (4, 780),
 (5, 1001),
 (5, 161),
 (5, 760),
 (5, 1400),
 (5, 1097),
 (5, 1355),
 (5, 443),
 (5, 1080),
 (5, 595),
 (5, 1227),
 (6, 143),
 (6, 664),
 (6, 1358),
 (6, 436),
 (6, 116),
 (6, 376),
 (6, 1312),
 (6, 246),
 (6, 606),
 (6, 302),
 (7, 711),
 (7, 1325),
 (7, 70),
 (7, 1080),
 (7, 1360),
 (7, 1336),
 (7, 649),
 (7, 837),
 (7, 986),
 (7, 760),
 (8, 700),
 (8, 346),
 (8, 88),
 (8, 1219),
 (8, 1091),
 (8, 98),
 (8, 759),
 (8, 131),
 (8, 1261),
 (8, 930),
 (9, 845),
 (9, 1172),
 (9, 711),
 (9, 558),
 (9, 814),
 (9, 172),
 (9, 1332),
 (9, 1490),
 (9, 591),
 (9, 986),
 (10

In [123]:
neighbor_candidates

[(1, 1308),
 (1, 2),
 (1, 244),
 (1, 847),
 (1, 943),
 (1, 4),
 (1, 1088),
 (1, 1283),
 (1, 18),
 (1, 359),
 (2, 4),
 (2, 18),
 (2, 704),
 (2, 1),
 (2, 780),
 (2, 127),
 (2, 226),
 (2, 1206),
 (2, 3),
 (2, 984),
 (3, 704),
 (3, 1380),
 (3, 258),
 (3, 1461),
 (3, 1014),
 (3, 518),
 (3, 847),
 (3, 147),
 (3, 1474),
 (3, 2),
 (4, 2),
 (4, 984),
 (4, 1330),
 (4, 1),
 (4, 1206),
 (4, 244),
 (4, 359),
 (4, 18),
 (4, 226),
 (4, 780),
 (5, 1001),
 (5, 161),
 (5, 760),
 (5, 1400),
 (5, 1097),
 (5, 1355),
 (5, 443),
 (5, 1080),
 (5, 595),
 (5, 1227),
 (6, 143),
 (6, 664),
 (6, 1358),
 (6, 436),
 (6, 116),
 (6, 376),
 (6, 1312),
 (6, 246),
 (6, 606),
 (6, 302),
 (7, 711),
 (7, 1325),
 (7, 70),
 (7, 1080),
 (7, 1360),
 (7, 1336),
 (7, 649),
 (7, 837),
 (7, 986),
 (7, 760),
 (8, 700),
 (8, 346),
 (8, 88),
 (8, 1219),
 (8, 1091),
 (8, 98),
 (8, 759),
 (8, 131),
 (8, 1261),
 (8, 930),
 (9, 845),
 (9, 1172),
 (9, 711),
 (9, 558),
 (9, 814),
 (9, 172),
 (9, 1332),
 (9, 1490),
 (9, 591),
 (9, 986),
 (10

In [112]:
def find_neighbors(centroids, radii, mol_meshes, kd, ids, box, neighbor_candidates):
    neighbor_pairs = []

    for pair in neighbor_candidates:
        i, j = pair
        if not blocked_by_any(i, j, centroids, radii, mol_meshes, kd, ids, box, cutoff_margin=2.0):
            neighbor_pairs.append((i, j))

        if i%10 == 0:
            print("Progress:", i)


    return neighbor_pairs

In [124]:
results = find_neighbors(centroids, radii, mol_meshes, kd, ids, box, neighbor_candidates)

results


Progress: 10
Progress: 10
Progress: 10
Progress: 10
Progress: 10
Progress: 10
Progress: 10
Progress: 10
Progress: 10
Progress: 10
Progress: 20
Progress: 20
Progress: 20
Progress: 20
Progress: 20
Progress: 20
Progress: 20
Progress: 20
Progress: 20
Progress: 20
Progress: 30
Progress: 30
Progress: 30
Progress: 30
Progress: 30
Progress: 30
Progress: 30
Progress: 30
Progress: 30
Progress: 30
Progress: 40
Progress: 40
Progress: 40
Progress: 40
Progress: 40
Progress: 40
Progress: 40
Progress: 40
Progress: 40
Progress: 40
Progress: 50
Progress: 50
Progress: 50
Progress: 50
Progress: 50
Progress: 50
Progress: 50
Progress: 50
Progress: 50
Progress: 50
Progress: 60
Progress: 60
Progress: 60
Progress: 60
Progress: 60
Progress: 60
Progress: 60
Progress: 60
Progress: 60
Progress: 60
Progress: 70
Progress: 70
Progress: 70
Progress: 70
Progress: 70
Progress: 70
Progress: 70
Progress: 70
Progress: 70
Progress: 70
Progress: 80
Progress: 80
Progress: 80
Progress: 80
Progress: 80
Progress: 80
Progress: 80

[(1, 847),
 (1, 4),
 (2, 18),
 (2, 3),
 (2, 984),
 (3, 258),
 (3, 1014),
 (3, 2),
 (4, 1330),
 (4, 1),
 (4, 226),
 (5, 760),
 (5, 1355),
 (5, 443),
 (6, 436),
 (6, 606),
 (8, 700),
 (8, 346),
 (8, 1219),
 (8, 1261),
 (9, 986),
 (10, 418),
 (10, 125),
 (11, 1067),
 (11, 73),
 (11, 1064),
 (12, 1126),
 (13, 571),
 (13, 992),
 (14, 62),
 (14, 873),
 (15, 788),
 (16, 1006),
 (17, 168),
 (17, 186),
 (17, 1116),
 (18, 2),
 (18, 127),
 (18, 231),
 (19, 1304),
 (19, 1107),
 (19, 612),
 (19, 46),
 (20, 795),
 (20, 1288),
 (21, 339),
 (21, 1466),
 (22, 327),
 (22, 833),
 (22, 195),
 (23, 931),
 (23, 373),
 (23, 801),
 (24, 108),
 (24, 1349),
 (25, 243),
 (25, 1278),
 (25, 1167),
 (25, 1472),
 (27, 796),
 (27, 33),
 (28, 706),
 (28, 1117),
 (28, 1112),
 (28, 229),
 (29, 1123),
 (29, 199),
 (30, 598),
 (30, 1124),
 (30, 299),
 (30, 118),
 (31, 1242),
 (32, 486),
 (32, 1054),
 (32, 1334),
 (32, 1090),
 (33, 580),
 (33, 387),
 (35, 1373),
 (36, 1188),
 (36, 536),
 (36, 809),
 (36, 1440),
 (37, 686),

In [126]:
len(results)

3097

In [ ]:
# from joblib import Parallel, delayed
# from scipy.spatial import cKDTree
# import numpy as np


# def find_neighbors_knn(mol_meshes, box, radii, centroids, ids, cutoff_margin=2.0, k=10, n_jobs=-1):
#     # Build KDTree once for blocked_by_any
#     ids = list(centroids.keys())
#     coords = np.vstack([centroids[i] for i in ids])
#     coords_wrapped = wrap_points(coords, box)
#     kd = cKDTree(coords_wrapped, boxsize=box)

#     # Get candidate neighbors from kNN
#     knn_dict = nearest_neighbors(mol_meshes, box, centroids, k=k, return_meshes=False)

#     def process_pair(i, j):
#         if not blocked_by_any(i, j, centroids, radii, mol_meshes, kd, ids, box, cutoff_margin):
#             return (i, j)
#         return None

#     # Prepare candidate pairs
#     tasks = [(i, j) for i, neigh_list in knn_dict.items() for j, _ in neigh_list if j > i]

#     print(f"There are {len(tasks)} possible neighbour pairs.")

#     # Run in parallel with progress bar
#     results = Parallel(n_jobs=n_jobs, batch_size=64)(
#         delayed(process_pair)(i, j) for (i, j) in tqdm(tasks, desc="Checking kNN neighbor pairs")
#     )

#     return [r for r in results if r is not None]
